# Lysosomal polygenic risk score analysis across GBA1-PD vs iPD/NMC/controls in GP2 Neurobooster genotyping data (all ancestries)

**Project:** GP2 lysosomal PRS

**Version:** Python/3.10.17, R/4.4.2

Notebook Overview

1. Description Loading Python libraries Set paths Make working directory
2. Installing packages
3. Process clinical data; Create a covariate file with GP2 data
4. Run PRSice 
    1. GBAPD VS NMC IN EUR
    2. GBAPD VS controls IN EUR
    3. GBAPD VS iPD IN EUR
5. Plot ROC
6. GLM analysis adjusting for sex, age, PC1-5

## Getting Started

### Import python dependencies

In [ ]:
## Import the necessary python dependencies 
%pip install seaborn --upgrade
!pip install rpy2
%load_ext rpy2.ipython
%pip install -U kaleido

from datetime import date
import importlib.metadata
from IPython.display import display
import math
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.collections import PatchCollection
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, TwoSlopeNorm
from matplotlib.ticker import FormatStrFormatter
import matplotlib.gridspec as gridspec
import numbers
import numpy as np
import os
import pandas as pd
import plotly.express as px
import requests
import scipy
from scipy import stats
from scipy.stats import norm
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import statsmodels.formula.api as smf
import subprocess
import sys
import seaborn as sns
import types
# Use pathlib for file path manipulation
import pathlib

today = date.today()
date = today.strftime("%d-%b-%Y").upper()

### Define helper functions

In [ ]:
def get_imports():
    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            # Split ensures you get root package, not just imported function
            name = val.__name__.split(".")[0]

        elif isinstance(val, type):
            name = val.__module__.split(".")[0]

        # Some packages are weird and have different imported names vs. system/pip names
        # Unfortunately, there is no systematic way to get pip names from a package's imported name. You'll have to add exceptions to this list manually!
        poorly_named_packages = {
            "PIL": "Pillow",
            "sklearn": "scikit-learn"
        }
        if name in poorly_named_packages.keys():
            name = poorly_named_packages[name]

        yield name

def min_max_scale(data):
    return (data - np.min(data)) / (np.max(data) - np.min(data))

def compare_rocs(input_path, output_path, auc1_variable, auc2_variable, xlabel):
    # Read p-values matrix and rescale values
    df_pvals = pd.read_csv(input_path, sep="\t", index_col="Ancestry")
    df_pvals_log = np.log(-np.log(np.abs(df_pvals)) + 1) * df_pvals / np.abs(df_pvals)
    
    # Prepare formatting for heatmap
    max_abs = np.max(np.abs(df_pvals_log))
    norm = TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs)
    annot = df_pvals.map(lambda x: "*" if -0.05 <= x <= 0.05 else "")
    
    # Generate heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(df_pvals_log, cmap="RdBu", norm=norm, cbar=True, ax=ax, annot=annot, fmt="")
    
    # Format color bar
    cbar = ax.collections[0].colorbar
    cbar.set_label("")
    cbar.set_ticks([-1.38522686, 1.38522686])
    cbar.set_ticklabels([f"{auc1_variable} Significantly Better", f"{auc2_variable} Significantly Better"])
    
    # Label axes and save figure
    plt.xlabel(xlabel)
    plt.ylabel("Target data ancestry")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

### Print out versions of imported python dependencies

In [ ]:
imports = list(set(get_imports()))
print(f"PACKAGE VERSIONS ({date})")
for m in importlib.metadata.distributions():
    if m.metadata["Name"] in imports and m.metadata["Name"]!="pip":
        print(f"\t{m.metadata['Name']}=={m.version}")

### Load R dependencies

In [ ]:
%%R

install.packages("caret")
install.packages("optparse", repos="https://cloud.r-project.org/")

In [ ]:
%%R

require(data.table)
require(dplyr)
require(ggplot2)

library(optparse)
library(data.table)
library("ggplot2")
library(RColorBrewer)
library("caret")
library("pROC")
install.packages("ggplot2")
library(ggplot2)

### Define directories and ancestry lists

In [ ]:
WORK_DIR = "/home/jupyter/workspace/ws_files/r11"
RESULTS_DIR = "/home/jupyter/workspace/ws_files/r11/results/aim2"
REL11_DIR = "/home/jupyter/workspace/gp2_tier2_eu_release11"

In [ ]:
%%R

WORK_DIR <- "/home/jupyter/workspace/ws_files/r11"
RESULTS_DIR <- "/home/jupyter/workspace/ws_files/r11/results/aim2"
REL11_DIR <- "/home/jupyter/workspace/gp2_tier2_eu_release11"

### Install bioinformatics packages

In [ ]:
%%bash

if test -e /home/jupyter/plink; then
    echo "Plink is already installed in /home/jupyter"
else
    echo "Plink is not installed"
    wget -P /home/jupyter http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 
    unzip -o /home/jupyter/plink_linux_x86_64_20190304.zip -d /home/jupyter
    rm /home/jupyter/plink_linux_x86_64_20190304.zip
fi

chmod u+x /home/jupyter/plink

In [ ]:
%%bash

if test -e /home/jupyter/plink2; then
    echo "Plink2 is already installed in /home/jupyter"
else
    echo "Plink2 is not installed"
    wget -P /home/jupyter http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip
    unzip -o /home/jupyter/plink2_linux_x86_64_latest.zip -d /home/jupyter
    rm /home/jupyter/plink2_linux_x86_64_latest.zip
fi

chmod u+x /home/jupyter/plink2

In [ ]:
%%bash

if test -e /home/jupyter/metal; then
    echo "Metal is already installed in /home/jupyter"
else
    echo "Metal is not installed"
    wget -P /home/jupyter https://csg.sph.umich.edu/abecasis/metal/download/Linux-metal.tar.gz
    tar --strip-components=1 -xzf /home/jupyter/Linux-metal.tar.gz -C /home/jupyter
    rm /home/jupyter/Linux-metal.tar.gz
fi

chmod u+x /home/jupyter/metal

In [ ]:
%%bash

if test -e /home/jupyter/prsice; then
    echo "PRSice is already installed in /home/jupyter"
else
    echo "PRSice is not installed"
    wget -P /home/jupyter https://github.com/choishingwan/PRSice/releases/download/2.3.5/PRSice_linux.zip
    unzip -o /home/jupyter/PRSice_linux.zip -d /home/jupyter
    mv /home/jupyter/PRSice_linux /home/jupyter/prsice
fi

chmod u+x /home/jupyter/prsice

## Process Clinical Data

### Generate covariates with PCs

In [ ]:
CLINICAL_DATA_PATH = pathlib.Path(REL11_DIR, 'clinical_data/master_key_release11_final_vwb.csv')

In [ ]:
# Let's load the master key
key = pd.read_csv(CLINICAL_DATA_PATH, low_memory=False)
print(key.shape)
key

In [ ]:
# Subsetting to keep only a few columns 
key = key[['GP2ID', 'baseline_GP2_phenotype_for_qc', 'biological_sex_for_qc', 'age_at_sample_collection', 'nba_label']]
# Renaming the columns
key.rename(columns = {'GP2ID':'IID',
                                     'baseline_GP2_phenotype_for_qc':'phenotype',
                                     'biological_sex_for_qc':'SEX', 
                                     'age_at_sample_collection':'AGE'}, inplace = True)
key

In [ ]:
# drop all NA values for all columns
key = key.dropna()
key

In [ ]:
no_mutations = pd.read_csv("/home/jupyter/workspace/ws_files/r11/results/covariate_IPD_rm.txt", sep='\s+')
no_mutations["Group"] = "no_mutations"
no_mutations = no_mutations[['FID', 'IID', 'Group', 'PHENO', 'SEX', 'AGE', 'ANCESTRY']]
no_mutations.rename(columns = {'PHENO':'phenotype',
                                     'ANCESTRY':'nba_label'}, inplace = True)
no_mutations

In [ ]:
no_mutations["phenotype"] = no_mutations["phenotype"].replace({
    1: "Control",
    2: "PD",
    -9: "Other"
})
no_mutations["SEX"] = no_mutations["SEX"].replace({
    1: "Male",
    2: "Female",
    0: "Other/Unknown/Not Reported"
})
no_mutations

In [ ]:
# filter out to only get eur
no_mutations = no_mutations[no_mutations["nba_label"].isin(["EUR"])]
no_mutations

In [ ]:
no_mutations["phenotype"].value_counts(dropna=False)

In [ ]:
# to filter out phenotype = 1 control IPD in EUR
no_mutations_controls_EUR = no_mutations[
    (no_mutations["phenotype"] == "Control")   # 1 = Control
]
no_mutations_controls_EUR

In [ ]:
# to filter out phenotype = 2 case IPD in EUR
IPD_EUR = no_mutations[
    (no_mutations["phenotype"] == "PD")   
]
IPD_EUR

In [ ]:
GBA_EUR = pd.read_csv("/home/jupyter/workspace/ws_files/r11/cohort/EUR/GBA1risk.samplestoKeep.rm.txt", sep='\t', header=None, names=['FID', 'IID'])
GBA_EUR = GBA_EUR.merge(key, on="IID", how="inner")
GBA_EUR

In [ ]:
GBA_EUR["phenotype"].value_counts(dropna=False)

In [ ]:
# Reformat sex column
GBA_EUR['SEX'] = GBA_EUR['SEX'].replace({
    'Female':2,
    'Male':1,
    'Other/Unknown/Not Reported':0,
})
GBA_EUR["phenotype"] = GBA_EUR["phenotype"].replace({
    "PD": 2,
    "Control": 1,
    "Other": -9,
})
display(GBA_EUR)

In [ ]:
df_pcs = pd.read_csv(f'{REL11_DIR}/meta_data/qc_metrics/projected_pcs_vwb.csv')
df_pcs = df_pcs[['IID', 'PC1', 'PC2','PC3', 'PC4','PC5']].copy()
df_pcs['IID'] = df_pcs['IID'].str.replace(r'_s1$', '', regex=True)
df_pcs

In [ ]:
# merge GBAPD_EUR with pcs
IPD_EUR = IPD_EUR.merge(df_pcs, on="IID")
IPD_EUR

In [ ]:
# merge IPD_controls_EUR with pcs
no_mutations_controls_EUR = no_mutations_controls_EUR.merge(df_pcs, on="IID")
no_mutations_controls_EUR

In [ ]:
GBA_EUR = GBA_EUR.merge(df_pcs, on="IID")
GBA_EUR

In [ ]:
GBA_EUR = GBA_EUR[GBA_EUR["phenotype"] != -9].copy()
GBA_EUR

In [ ]:
# to filter GBA_EUR phenotype = 2
GBAPD_EUR = GBA_EUR[GBA_EUR["phenotype"] == 2].copy()
GBAPD_EUR

In [ ]:
# Rename columns to match desired output
IPD_EUR = IPD_EUR.rename(columns={
    'nba_label': 'ANCESTRY',
    'phenotype': 'PHENO'
})

# Reorder the columns
columns_order = ['FID', 'IID', 'SEX', 'AGE', 'ANCESTRY', 'PHENO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
IPD_EUR = IPD_EUR[columns_order]
IPD_EUR

In [ ]:
# Reformat sex column
IPD_EUR['SEX'] = IPD_EUR['SEX'].replace({
    'Female':2,
    'Male':1,
    'Other/Unknown/Not Reported':0,
})
IPD_EUR["PHENO"] = IPD_EUR["PHENO"].replace({
    "PD": 2,
    "Control": 1,
    "Other": -9,
})
display(IPD_EUR)

In [ ]:
# Rename columns to match desired output
GBAPD_EUR = GBAPD_EUR.rename(columns={
    'nba_label': 'ANCESTRY',
    'phenotype': 'PHENO'
})

# Reorder the columns
columns_order = ['FID', 'IID', 'SEX', 'AGE', 'ANCESTRY', 'PHENO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
GBAPD_EUR = GBAPD_EUR[columns_order]
GBAPD_EUR

In [ ]:
# Rename columns to match desired output
no_mutations_controls_EUR = no_mutations_controls_EUR.rename(columns={
    'nba_label': 'ANCESTRY',
    'phenotype': 'PHENO'
})

# Reorder the columns
columns_order = ['FID', 'IID', 'SEX', 'AGE', 'ANCESTRY', 'PHENO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
no_mutations_controls_EUR = no_mutations_controls_EUR[columns_order]
no_mutations_controls_EUR

In [ ]:
# Reformat sex column
no_mutations_controls_EUR['SEX'] = no_mutations_controls_EUR['SEX'].replace({
    'Female':2,
    'Male':1,
    'Other/Unknown/Not Reported':0,
})
no_mutations_controls_EUR["PHENO"] = no_mutations_controls_EUR["PHENO"].replace({
    "PD": 2,
    "Control": 1,
    "Other": -9,
})
display(no_mutations_controls_EUR)

In [ ]:
# Stack rows (controls + cases)
GBAPD_EUR_HC = pd.concat([GBAPD_EUR, no_mutations_controls_EUR], ignore_index=True)
GBAPD_EUR_HC

In [ ]:
GBAPD_EUR_HC["PHENO"].value_counts(dropna=False)

In [ ]:
GBAPD_EUR_HC_samplestokeep = GBAPD_EUR_HC[["FID", "IID"]].copy()
GBAPD_EUR_HC_samplestokeep.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_HC_samplestokeep.txt", index=False, sep="\t")

In [ ]:
GBAPD_EUR_HC.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_HC_covariate.txt", index=False, sep="\t")

In [ ]:
GBAPD_EUR_HC = GBAPD_EUR_HC[["FID","IID","SEX","AGE","PC1","PC2","PC3","PC4","PC5"]]
GBAPD_EUR_HC

In [ ]:
# Save to a .txt file contain control from IPD category
GBAPD_EUR_HC.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_HC.txt", sep='\t', index=False)

In [ ]:
# to filter GBA_EUR phenotype = 2
NMC = GBA_EUR[GBA_EUR["phenotype"] == 1].copy()
NMC

In [ ]:
# Rename columns to match desired output
NMC = NMC.rename(columns={
    'nba_label': 'ANCESTRY',
    'phenotype': 'PHENO'
})

# Reorder the columns
columns_order = ['FID', 'IID', 'SEX', 'AGE', 'ANCESTRY', 'PHENO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
NMC = NMC[columns_order]
NMC

In [ ]:
# Stack rows (controls + cases)
GBAPD_EUR_NMC = pd.concat([GBAPD_EUR, NMC], ignore_index=True)
GBAPD_EUR_NMC

In [ ]:
GBAPD_EUR_NMC["PHENO"].value_counts(dropna=False)

In [ ]:
GBAPD_EUR_NMC_samplestokeep = GBAPD_EUR_NMC[["FID", "IID"]].copy()
GBAPD_EUR_NMC_samplestokeep.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_NMC_samplestokeep.txt", index=False, sep="\t")

In [ ]:
GBAPD_EUR_NMC.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_NMC_covariate.txt", index=False, sep="\t")

In [ ]:
GBAPD_EUR_NMC = GBAPD_EUR_NMC[["FID","IID","SEX","AGE","PC1","PC2","PC3","PC4","PC5"]]
GBAPD_EUR_NMC

In [ ]:
# Save to a .txt file contain control from IPD category
GBAPD_EUR_NMC.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_NMC.txt", sep='\t', index=False)

In [ ]:
GBAPD_EUR

In [ ]:
IPD_EUR

In [ ]:
# Make phenotype file for PRS: GBA1-PD cases vs iPD controls

# GBA1-PD cases
GBAPD_EUR_pheno = GBAPD_EUR[['FID', 'IID']].copy()
GBAPD_EUR_pheno['PHENO'] = 2

# iPD controls
IPD_EUR_pheno = IPD_EUR[['FID', 'IID']].copy()
IPD_EUR_pheno['PHENO'] = 1

# Combine cases and controls
GBAPD_EUR_IPD_PHENO = pd.concat(
    [GBAPD_EUR_pheno, IPD_EUR_pheno],
    ignore_index=True
)

# Check phenotype counts
GBAPD_EUR_IPD_PHENO['PHENO'].value_counts()

In [ ]:
GBAPD_EUR_IPD_PHENO.to_csv(
    "/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD_PHENO.txt",
    sep="\t",
    index=False
)

In [ ]:
# Stack rows (controls + cases)
GBAPD_EUR_IPD = pd.concat([GBAPD_EUR, IPD_EUR], ignore_index=True)
GBAPD_EUR_IPD

In [ ]:
GBAPD_EUR_IPD["PHENO"].value_counts(dropna=False)

In [ ]:
GBAPD_EUR_IPD_samplestokeep = GBAPD_EUR_IPD[["FID", "IID"]].copy()
GBAPD_EUR_IPD_samplestokeep.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD_samplestokeep.txt", index=False, sep="\t")

In [ ]:
GBAPD_EUR_IPD.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD_covariate.txt", index=False, sep="\t")

In [ ]:
GBAPD_EUR_IPD = GBAPD_EUR_IPD[["FID","IID","SEX","AGE","PC1","PC2","PC3","PC4","PC5"]]
GBAPD_EUR_IPD

In [ ]:
# Save to a .txt file contain control from IPD category
GBAPD_EUR_IPD.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD.txt", sep='\t', index=False)

## GBAPD VS NMC IN EUR

### Run PRSice (GBAPD VS NMC IN EUR)

In [ ]:
# GBAPD VS HCcarriers IN EUR
# the prevalence of GBA1-PD in GBA1 carriers
# cycle using SNP list without GBA and LRRK2
# remove --score con-std \
! Rscript /home/jupyter/PRSice.R \
--out /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HCcarriers_EUR/GBA_HCcarriers_raw \
--target /home/jupyter/workspace/ws_files/r11/genotype_file/EUR/chr# \
-b /home/jupyter/workspace/ws_files/SNPlist_MsigDB/SNP_list_EUR_noLRRK2GBA.txt \
--beta \
--snp SNP \
--a1 A1 \
--a2 A2 \
--stat BETA \
--pvalue P \
--ld /home/jupyter/workspace/ws_files/reference/ref_by_pop/all_hg38_EUR \
--print-snp \
--perm 10000 \
--bar-levels 0.1 \
--prsice /home/jupyter/prsice \
-n 24 \
--binary-target T \
--quantile 4 \
--no-full \
--fastscore \
--prevalence 0.05 \
--thread 16 \
--keep /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_NMC_samplestokeep.txt \
--cov-file /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_NMC.txt

### Estimate Specificity and Sensitivity

In [ ]:
%%R

dat <- read.table(paste0(RESULTS_DIR, "/", "GBAPD_HCcarriers_EUR/", "GBA_HCcarriers_raw", ".best"), header = TRUE, sep = " ")
cov <- read.table(paste0(RESULTS_DIR, "/GBAPD_EUR_NMC_covariate.txt"), header = TRUE, sep = "\t")
colnames(cov) <- c("FID", "IID", "SEX", "AGE", "ANCESTRY", "PHENO", "PC1", "PC2", "PC3", "PC4", "PC5")
dat <- merge(dat, cov, by = "IID")
dat$CASE <- dat$PHENO - 1
dat <- subset(dat, CASE != -10)
meanControls <- mean(dat$PRS[dat$CASE == 0])
sdControls <- sd(dat$PRS[dat$CASE == 0])
dat$zSCORE <- (dat$PRS - meanControls) / sdControls
grsTests <- glm(CASE ~ zSCORE, family = "binomial", data = dat)
dat$probDisease <- predict(grsTests, dat, type = "response")
dat$reported <- ifelse(dat$CASE == 1, "DISEASE", "CONTROL")
roc <- roc(response = dat$reported, predictor = dat$probDisease)
auc_value <- auc(roc)

result.coords <- coords(
    roc, 
    "best", 
    best.method = "closest.topleft", 
    ret = c("threshold", "accuracy", "specificity", "sensitivity", "youden"),
    )
dat$predicted <- ifelse(dat$probDisease > result.coords$threshold, "DISEASE", "CONTROL")
confMat <- confusionMatrix(data = as.factor(dat$predicted), reference = as.factor(dat$reported), positive = "DISEASE")

results_dt <- data.table(
  Ancestry = "EUR",
  AUC = auc_value,
  Accuracy = confMat$overall["Accuracy"],
  CI_Lower = confMat$overall["AccuracyLower"],
  CI_Upper = confMat$overall["AccuracyUpper"],
  Balanced_Accuracy = confMat$byClass["Balanced Accuracy"],
  Sensitivity = confMat$byClass["Sensitivity"],
  Specificity = confMat$byClass["Specificity"]
)

fwrite(results_dt, paste0(RESULTS_DIR, "/GBAPD_HCcarriers_EUR/GBAPD_HCcarriers_EUR_r11_results_raw.txt"), sep = "\t", quote = FALSE, row.names = FALSE)
print(results_dt)

In [ ]:
df_bestfit = pd.read_csv(f"{RESULTS_DIR}/GBAPD_HCcarriers_EUR/GBAPD_HCcarriers_EUR_r11_results_raw.txt", sep="\s+")
df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]] = df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]].round(3).astype(str)
df_bestfit["Accuracy"] = df_bestfit.apply(lambda row: f"{row['Accuracy']} ({row['CI_Lower']} - {row['CI_Upper']})", axis=1)
df_bestfit.rename(columns={"Accuracy":"Accuracy (95% CI)", "Balanced_Accuracy":"Balanced Accuracy"}, inplace=True)
df_bestfit.drop(["CI_Lower", "CI_Upper"], inplace=True, axis=1)


df = pd.read_csv(f"{RESULTS_DIR}/GBAPD_HCcarriers_EUR/GBA_HCcarriers_raw.summary", sep="\s+")
df["Ancestry"] = "EUR"  
df = df[[
    "Ancestry",
    "Threshold",
    "PRS.R2",
    "Full.R2",
    "Null.R2",
    "Coefficient",
    "Standard.Error",
    "Num_SNP",
]]
df.rename(columns={
    "PRS.R2":"PRS R2",
    "Full.R2":"Full R2",
    "Null.R2":"Null R2",
    "Standard.Error":"SE",
    "Num_SNP":"No. of SNP",
}, inplace=True)
lower = f"{math.e ** (df.loc[0,'Coefficient'] - 1.96 * df.loc[0,'SE']):.2f}"
upper = f"{math.e ** (df.loc[0,'Coefficient'] + 1.96 * df.loc[0,'SE']):.2f}"
df["OR (95% CI)"] = f"{math.e ** df.loc[0,'Coefficient']:.2f} ({lower} - {upper})"
df_prsice_list = []
df_prsice_list.append(df)
df_prsice = pd.concat(df_prsice_list, ignore_index=True)

df_prsice[["PRS R2","Full R2","Null R2","Coefficient","SE"]] = df_prsice[["PRS R2","Full R2","Null R2","Coefficient","SE"]].round(3)
df_prsice["Threshold"] = df_prsice["Threshold"].apply(lambda x: f'{x:.2e}')

df_merged = df_prsice.merge(df_bestfit, on="Ancestry")
df_merged.to_csv(f"{RESULTS_DIR}/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11.table_raw.txt", index=False, sep="\t")

### Plot ROC

In [ ]:
%%R
prs_data <- fread(paste0(RESULTS_DIR, '/','/GBAPD_HCcarriers_EUR/', 'GBA_HCcarriers_raw.best'), header=T)
colnames(prs_data)[4] <- "PRS"
covar <- fread(paste0(RESULTS_DIR, "/GBAPD_EUR_NMC_covariate.txt"), header=T)
colnames(covar)[1] <- "FID"
colnames(covar)[2] <- "IID"
temp <- merge(prs_data, covar, by = c("FID","IID"))
temp$CASE <- temp$PHENO - 1
DATA <- subset(temp, CASE != -10)

## Probability of disease calculation
Model <- glm(CASE ~ PRS, data = DATA, family = "binomial")
DATA$probDisease <- predict(Model, DATA, type = c("response"))
DATA$predicted <- ifelse(DATA$probDisease > 0.5, "DISEASE", "CONTROL")
DATA$reported <- ifelse(DATA$CASE == 1, "DISEASE","CONTROL")

write.table(DATA, file = paste0(RESULTS_DIR, "/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11_risk_results_raw.txt"), sep = "\t", quote = FALSE, row.names = FALSE)

In [ ]:
df = pd.read_csv(f"{RESULTS_DIR}/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11_risk_results_raw.txt", sep="\t")
fpr, tpr, _ = roc_curve(df["CASE"], df["probDisease"])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 5))
sns.set_palette("Dark2")
plt.plot(fpr, tpr, label=f"GBA1-PD vs NMC_EUR (AUC = {roc_auc:.2f})", lw=2)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--') 

plt.title("ROC Curve - GBA1-PD vs NMC in EUR")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()

plt.savefig(f"{RESULTS_DIR}/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11_ROC_raw.png", dpi=300)
plt.close()

### Regression analysis

In [ ]:
%%R
# Read tab-delimited file
data <- fread("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11_risk_results_raw.txt")
head(data)

In [ ]:
%%R
# save into csv
fwrite(data, "/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11_risk_results_raw.csv")

In [ ]:
%%R
table(data$PHENO)
data$pheno_label <- ifelse(data$PHENO == 2, "Case", "Control")
head(data)

In [ ]:
%%R
# Ensure PHENO is binary (0 = Control, 1 = Case)
data$PHENO <- ifelse(data$PHENO == 2, 1, 0)

In [ ]:
%%R
# Logistic regression
model <- glm(PHENO ~ PRS + SEX + AGE + PC1 + PC2 + PC3 + PC4 + PC5, family = "binomial", data = data)

summary_model <- coef(summary(model))
summary_model <- as.data.frame(summary_model)
summary_model$Variable <- rownames(summary_model)
rownames(summary_model) <- NULL
print(summary_model)  # Shows p-values, std error, etc.

In [ ]:
%%R
write.table(summary_model, file = "/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HCcarriers_EUR/GBA_HCcarriers_logistic_results_raw.txt",
            sep = "\t", row.names = FALSE, quote = FALSE)

## GBAPD vs HC in EUR

### Run PRSice (GBAPD VS HC IN EUR)

In [ ]:
# GBA VS HC IN EUR
# GBA-PD prevalence in the general population
# --score con-std \
! Rscript /home/jupyter/PRSice.R \
--out /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HC_EUR/GBA_HC_EUR_r11_raw \
--target /home/jupyter/workspace/ws_files/r11/genotype_file/EUR/chr# \
-b /home/jupyter/workspace/ws_files/SNPlist_MsigDB/SNP_list_EUR_noLRRK2GBA.txt \
--beta \
--snp SNP \
--a1 A1 \
--a2 A2 \
--stat BETA \
--pvalue P \
--ld /home/jupyter/workspace/ws_files/reference/ref_by_pop/all_hg38_EUR \
--print-snp \
--bar-levels 0.1 \
--fastscore \
--perm 10000 \
--prevalence 0.00005 \
--prsice /home/jupyter/prsice \
-n 24 \
--binary-target T \
--quantile 4 \
--no-full \
--thread 16 \
--keep /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_HC_samplestokeep.txt \
--cov-file /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_HC.txt

### Estimate Specificity and Sensitivity

In [ ]:
RESULTS_DIR

In [ ]:
%%R
dat <- read.table(paste0(RESULTS_DIR, "/", "GBAPD_HC_EUR/", "GBA_HC_EUR_r11_raw", ".best"), header = TRUE, sep = " ")
cov <- read.table(paste0(RESULTS_DIR, "/GBAPD_EUR_HC_covariate.txt"), header = TRUE, sep = "\t")
colnames(cov) <- c("FID", "IID", "SEX", "AGE", "ANCESTRY", "PHENO", "PC1", "PC2", "PC3", "PC4", "PC5")
dat <- merge(dat, cov, by = "IID")
dat$CASE <- dat$PHENO - 1
dat <- subset(dat, CASE != -10)
meanControls <- mean(dat$PRS[dat$CASE == 0])
sdControls <- sd(dat$PRS[dat$CASE == 0])
dat$zSCORE <- (dat$PRS - meanControls) / sdControls
grsTests <- glm(CASE ~ zSCORE, family = "binomial", data = dat)
dat$probDisease <- predict(grsTests, dat, type = "response")
dat$reported <- ifelse(dat$CASE == 1, "DISEASE", "CONTROL")
roc <- roc(response = dat$reported, predictor = dat$probDisease)
auc_value <- auc(roc)

result.coords <- coords(
    roc, 
    "best", 
    best.method = "closest.topleft", 
    ret = c("threshold", "accuracy", "specificity", "sensitivity", "youden"),
    )
dat$predicted <- ifelse(dat$probDisease > result.coords$threshold, "DISEASE", "CONTROL")
confMat <- confusionMatrix(data = as.factor(dat$predicted), reference = as.factor(dat$reported), positive = "DISEASE")

results_dt <- data.table(
  Ancestry = "EUR",
  AUC = auc_value,
  Accuracy = confMat$overall["Accuracy"],
  CI_Lower = confMat$overall["AccuracyLower"],
  CI_Upper = confMat$overall["AccuracyUpper"],
  Balanced_Accuracy = confMat$byClass["Balanced Accuracy"],
  Sensitivity = confMat$byClass["Sensitivity"],
  Specificity = confMat$byClass["Specificity"]
)

fwrite(results_dt, paste0(RESULTS_DIR, "/GBAPD_HC_EUR/GBA_HC_EUR_r11_results_raw.txt"), sep = "\t", quote = FALSE, row.names = FALSE)
print(results_dt)

In [ ]:
df_bestfit = pd.read_csv(f"{RESULTS_DIR}/GBAPD_HC_EUR/GBA_HC_EUR_r11_results_raw.txt", sep="\s+")
df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]] = df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]].round(3).astype(str)
df_bestfit["Accuracy"] = df_bestfit.apply(lambda row: f"{row['Accuracy']} ({row['CI_Lower']} - {row['CI_Upper']})", axis=1)
df_bestfit.rename(columns={"Accuracy":"Accuracy (95% CI)", "Balanced_Accuracy":"Balanced Accuracy"}, inplace=True)
df_bestfit.drop(["CI_Lower", "CI_Upper"], inplace=True, axis=1)


df = pd.read_csv(f"{RESULTS_DIR}/GBAPD_HC_EUR/GBA_HC_EUR_r11_raw.summary", sep="\s+")
df["Ancestry"] = "EUR"  
df = df[[
    "Ancestry",
    "Threshold",
    "PRS.R2",
    "Full.R2",
    "Null.R2",
    "Coefficient",
    "Standard.Error",
    "Num_SNP",
]]
df.rename(columns={
    "PRS.R2":"PRS R2",
    "Full.R2":"Full R2",
    "Null.R2":"Null R2",
    "Standard.Error":"SE",
    "Num_SNP":"No. of SNP",
}, inplace=True)
lower = f"{math.e ** (df.loc[0,'Coefficient'] - 1.96 * df.loc[0,'SE']):.2f}"
upper = f"{math.e ** (df.loc[0,'Coefficient'] + 1.96 * df.loc[0,'SE']):.2f}"
df["OR (95% CI)"] = f"{math.e ** df.loc[0,'Coefficient']:.2f} ({lower} - {upper})"
df_prsice_list = []
df_prsice_list.append(df)
df_prsice = pd.concat(df_prsice_list, ignore_index=True)

df_prsice[["PRS R2","Full R2","Null R2","Coefficient","SE"]] = df_prsice[["PRS R2","Full R2","Null R2","Coefficient","SE"]].round(3)
df_prsice["Threshold"] = df_prsice["Threshold"].apply(lambda x: f'{x:.2e}')

df_merged = df_prsice.merge(df_bestfit, on="Ancestry")
df_merged.to_csv(f"{RESULTS_DIR}/GBAPD_HC_EUR/GBA_HC_EUR_r11.table_raw.txt", index=False, sep="\t")

### Plot ROC

In [ ]:
%%R
prs_data <- fread(paste0(RESULTS_DIR, '/GBAPD_HC_EUR/', 'GBA_HC_EUR_r11_raw.best'), header=T)
colnames(prs_data)[4] <- "PRS"
covar <- fread(paste0(RESULTS_DIR, "/GBAPD_EUR_HC_covariate.txt"), header=T)
colnames(covar)[1] <- "FID"
colnames(covar)[2] <- "IID"
temp <- merge(prs_data, covar, by = c("FID","IID"))
temp$CASE <- temp$PHENO - 1
DATA <- subset(temp, CASE != -10)

## Probability of disease calculation
Model <- glm(CASE ~ PRS, data = DATA, family = "binomial")
DATA$probDisease <- predict(Model, DATA, type = c("response"))
DATA$predicted <- ifelse(DATA$probDisease > 0.5, "DISEASE", "CONTROL")
DATA$reported <- ifelse(DATA$CASE == 1, "DISEASE","CONTROL")

write.table(DATA, file = paste0(RESULTS_DIR, "/GBAPD_HC_EUR/GBA_HC_EUR_r11_risk_results_raw.txt"), sep = "\t", quote = FALSE, row.names = FALSE)

In [ ]:
df = pd.read_csv(f"{RESULTS_DIR}/GBAPD_HC_EUR/GBA_HC_EUR_r11_risk_results_raw.txt", sep="\t")
fpr, tpr, _ = roc_curve(df["CASE"], df["probDisease"])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 5))
sns.set_palette("Dark2")
plt.plot(fpr, tpr, label=f"GBA1-PD vs HC_EUR (AUC = {roc_auc:.2f})", lw=2)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--') 

plt.title("ROC Curve - GBA1-PD vs HC in EUR")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()

plt.savefig(f"{RESULTS_DIR}/GBAPD_HC_EUR/GBA_HC_EUR_r11_ROC_raw.png", dpi=300)
plt.close()

### Regression analysis

In [ ]:
%%R
# Read tab-delimited file
data <- fread("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HC_EUR/GBA_HC_EUR_r11_risk_results_raw.txt")
head(data)

In [ ]:
%%R
# save into csv
fwrite(data, "/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HC_EUR/GBA_HC_r11_risk_results_raw.csv")

In [ ]:
%%R
table(data$PHENO)
data$pheno_label <- ifelse(data$PHENO == 2, "Case", "Control")
head(data)

In [ ]:
%%R
# Ensure PHENO is binary (0 = Control, 1 = Case)
data$PHENO <- ifelse(data$PHENO == 2, 1, 0)

In [ ]:
%%R
# Logistic regression
model <- glm(PHENO ~ PRS + SEX + AGE + PC1 + PC2 + PC3 + PC4 + PC5, family = "binomial", data = data)

summary_model <- coef(summary(model))
summary_model <- as.data.frame(summary_model)
summary_model$Variable <- rownames(summary_model)
rownames(summary_model) <- NULL
print(summary_model)  # Shows p-values, std error, etc.

In [ ]:
%%R
write.table(summary_model, file = "/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_HC_EUR/GBA_HC_logistic_results_raw.txt",
            sep = "\t", row.names = FALSE, quote = FALSE)

## GBAPD VS IPD in EUR

### Run PRSice (GBAPD VS IPD IN EUR)

In [ ]:
# GBAPD VS IPD IN EUR
# GBA1-PD prevalence in PD
# cycle using SNP list without GBA and LRRK2 v1 with prevalence
# --score con-std \
! Rscript /home/jupyter/PRSice.R \
--out /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_IPD/GBAPD_IPD_raw \
--target /home/jupyter/workspace/ws_files/r11/genotype_file/EUR/chr# \
-b /home/jupyter/workspace/ws_files/SNPlist_MsigDB/SNP_list_EUR_noLRRK2GBA.txt \
--beta \
--snp SNP \
--a1 A1 \
--a2 A2 \
--stat BETA \
--pvalue P \
--ld /home/jupyter/workspace/ws_files/reference/ref_by_pop/all_hg38_EUR \
--print-snp \
--perm 10000 \
--prsice /home/jupyter/prsice \
-n 24 \
--binary-target T \
--quantile 4 \
--fastscore \
--no-full \
--thread 16 \
--prevalence 0.1 \
--bar-levels 0.1 \
--pheno-file /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD_PHENO.txt \
--pheno-col PHENO \
--keep /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD_samplestokeep.txt \
--cov-file /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD.txt

In [ ]:
! head /home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD_PHENO.txt

### Estimate Specificity and Sensitivity

In [ ]:
RESULTS_DIR

In [ ]:
%%R

head(dat)

In [ ]:
%%R

head(cov)

In [ ]:
# GBA1-PD cases
GBAPD_EUR_case = GBAPD_EUR.copy()
GBAPD_EUR_case["PHENO"] = 2

# iPD controls
IPD_EUR_control = IPD_EUR.copy()
IPD_EUR_control["PHENO"] = 1

# combine
GBAPD_EUR_IPD_covariate = pd.concat(
    [GBAPD_EUR_case, IPD_EUR_control],
    ignore_index=True
)

# check
GBAPD_EUR_IPD_covariate["PHENO"].value_counts()

In [ ]:
GBAPD_EUR_IPD_covariate.to_csv("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_EUR_IPD_covariate.txt", index=False, sep="\t")

In [ ]:
%%R

dat <- read.table(paste0(RESULTS_DIR, "/", "GBAPD_IPD/", "GBAPD_IPD_raw", ".best"), header = TRUE, sep = " ")
cov <- read.table(paste0(RESULTS_DIR, "/GBAPD_EUR_IPD_covariate.txt"), header = TRUE, sep = "\t")
colnames(cov) <- c("FID", "IID", "SEX", "AGE", "ANCESTRY", "PHENO", "PC1", "PC2", "PC3", "PC4", "PC5")
dat <- merge(dat, cov, by = "IID")
dat$CASE <- dat$PHENO - 1
dat <- subset(dat, CASE != -10)
meanControls <- mean(dat$PRS[dat$CASE == 0])
sdControls <- sd(dat$PRS[dat$CASE == 0])
dat$zSCORE <- (dat$PRS - meanControls) / sdControls
grsTests <- glm(CASE ~ zSCORE, family = "binomial", data = dat)
dat$probDisease <- predict(grsTests, dat, type = "response")
dat$reported <- ifelse(dat$CASE == 1, "DISEASE", "CONTROL")
roc <- roc(response = dat$reported, predictor = dat$probDisease)
auc_value <- auc(roc)

result.coords <- coords(
    roc, 
    "best", 
    best.method = "closest.topleft", 
    ret = c("threshold", "accuracy", "specificity", "sensitivity", "youden"),
    )
dat$predicted <- ifelse(dat$probDisease > result.coords$threshold, "DISEASE", "CONTROL")
confMat <- confusionMatrix(data = as.factor(dat$predicted), reference = as.factor(dat$reported), positive = "DISEASE")

results_dt <- data.table(
  Ancestry = "EUR",
  AUC = auc_value,
  Accuracy = confMat$overall["Accuracy"],
  CI_Lower = confMat$overall["AccuracyLower"],
  CI_Upper = confMat$overall["AccuracyUpper"],
  Balanced_Accuracy = confMat$byClass["Balanced Accuracy"],
  Sensitivity = confMat$byClass["Sensitivity"],
  Specificity = confMat$byClass["Specificity"]
)

fwrite(results_dt, paste0(RESULTS_DIR, "/GBAPD_IPD/GBAPD_IPD_r11_results_raw.txt"), sep = "\t", quote = FALSE, row.names = FALSE)
print(results_dt)

In [ ]:
df_bestfit = pd.read_csv(f"{RESULTS_DIR}/GBAPD_IPD/GBAPD_IPD_r11_results_raw.txt", sep="\s+")
df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]] = df_bestfit[["AUC","Accuracy","CI_Lower","CI_Upper","Balanced_Accuracy","Sensitivity","Specificity"]].round(3).astype(str)
df_bestfit["Accuracy"] = df_bestfit.apply(lambda row: f"{row['Accuracy']} ({row['CI_Lower']} - {row['CI_Upper']})", axis=1)
df_bestfit.rename(columns={"Accuracy":"Accuracy (95% CI)", "Balanced_Accuracy":"Balanced Accuracy"}, inplace=True)
df_bestfit.drop(["CI_Lower", "CI_Upper"], inplace=True, axis=1)

df = pd.read_csv(f"{RESULTS_DIR}/GBAPD_IPD/GBAPD_IPD_raw.summary", sep="\s+")
df["Ancestry"] = "EUR"  
df = df[[
    "Ancestry",
    "Threshold",
    "PRS.R2",
    "Full.R2",
    "Null.R2",
    "Coefficient",
    "Standard.Error",
    "Num_SNP",
]]
df.rename(columns={
    "PRS.R2":"PRS R2",
    "Full.R2":"Full R2",
    "Null.R2":"Null R2",
    "Standard.Error":"SE",
    "Num_SNP":"No. of SNP",
}, inplace=True)
lower = f"{math.e ** (df.loc[0,'Coefficient'] - 1.96 * df.loc[0,'SE']):.2f}"
upper = f"{math.e ** (df.loc[0,'Coefficient'] + 1.96 * df.loc[0,'SE']):.2f}"
df["OR (95% CI)"] = f"{math.e ** df.loc[0,'Coefficient']:.2f} ({lower} - {upper})"
df_prsice_list = []
df_prsice_list.append(df)
df_prsice = pd.concat(df_prsice_list, ignore_index=True)

df_prsice[["PRS R2","Full R2","Null R2","Coefficient","SE"]] = df_prsice[["PRS R2","Full R2","Null R2","Coefficient","SE"]].round(3)
df_prsice["Threshold"] = df_prsice["Threshold"].apply(lambda x: f'{x:.2e}')

df_merged = df_prsice.merge(df_bestfit, on="Ancestry")
df_merged.to_csv(f"{RESULTS_DIR}/GBAPD_IPD/GBAPD_IPD_r11.table_raw.txt", index=False, sep="\t")

### Plot ROC

In [ ]:
%%R
prs_data <- fread(paste0(RESULTS_DIR, '/','/GBAPD_IPD/', 'GBAPD_IPD_raw.best'), header=T)
colnames(prs_data)[4] <- "PRS"
covar <- fread(paste0(RESULTS_DIR, "/GBAPD_EUR_IPD_covariate.txt"), header=T)
colnames(covar)[1] <- "FID"
colnames(covar)[2] <- "IID"
temp <- merge(prs_data, covar, by = c("FID","IID"))
temp$CASE <- temp$PHENO - 1
DATA <- subset(temp, CASE != -10)

## Probability of disease calculation
Model <- glm(CASE ~ PRS, data = DATA, family = "binomial")
DATA$probDisease <- predict(Model, DATA, type = c("response"))
DATA$predicted <- ifelse(DATA$probDisease > 0.5, "DISEASE", "CONTROL")
DATA$reported <- ifelse(DATA$CASE == 1, "DISEASE","CONTROL")

write.table(DATA, file = paste0(RESULTS_DIR, "/GBAPD_IPD/GBAPD_IPD_r11_risk_results_raw.txt"), sep = "\t", quote = FALSE, row.names = FALSE)

In [ ]:
df = pd.read_csv(f"{RESULTS_DIR}/GBAPD_IPD/GBAPD_IPD_r11_risk_results_raw.txt", sep="\t")
fpr, tpr, _ = roc_curve(df["CASE"], df["probDisease"])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 5))
sns.set_palette("Dark2")
plt.plot(fpr, tpr, label=f"GBA1-PD vs iPD (AUC = {roc_auc:.2f})", lw=2)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--') 

plt.title("ROC Curve - GBA1-PD vs iPD in EUR")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()

plt.savefig(f"{RESULTS_DIR}/GBAPD_IPD/GBAPD_IPD_r11_ROC_raw.png", dpi=300)
plt.close()

### Regression analysis

In [ ]:
%%R
# Read tab-delimited file
data <- fread("/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_IPD/GBAPD_IPD_r11_risk_results_raw.txt")
head(data)

In [ ]:
%%R
# save into csv
fwrite(data, "/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_IPD/GBAPD_IPD_r11_risk_results_raw.csv")

In [ ]:
%%R
table(data$PHENO)
data$pheno_label <- ifelse(data$PHENO == 2, "Case", "Control")
head(data)

In [ ]:
%%R
# Ensure PHENO is binary (0 = Control, 1 = Case)
data$PHENO <- ifelse(data$PHENO == 2, 1, 0)

In [ ]:
%%R
# Logistic regression
model <- glm(PHENO ~ PRS + SEX + AGE + PC1 + PC2 + PC3 + PC4 + PC5, family = "binomial", data = data)

summary_model <- coef(summary(model))
summary_model <- as.data.frame(summary_model)
summary_model$Variable <- rownames(summary_model)
rownames(summary_model) <- NULL
print(summary_model)  # Shows p-values, std error, etc.

In [ ]:
%%R
write.table(summary_model, file = "/home/jupyter/workspace/ws_files/r11/results/aim2/GBAPD_IPD/GBAPD_IPD_logistic_results_raw.txt",
            sep = "\t", row.names = FALSE, quote = FALSE)

## Visualizations - Plot ROC Curves Together

In [ ]:
# visualization
# make ROC curves together in one plot

def plot_roc(ax, filepath, label, y="CASE", score="probDisease", lw=2):
    df = pd.read_csv(filepath, sep="\t")
    fpr, tpr, _ = roc_curve(df[y], df[score])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=lw, label=f"{label} (AUC = {roc_auc:.2f})")
    return roc_auc

# --- one plot ---
fig, ax = plt.subplots(figsize=(8, 5))
sns.set_palette("Dark2")  # optional styling

# diagonal
ax.plot([0, 1], [0, 1], color="gray", linestyle="--")

# add each ROC curve
plot_roc(
    ax,
    f"{RESULTS_DIR}/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11_risk_results_raw.txt",
    label="GBA1-PD vs NMC(EUR)"
)

plot_roc(
    ax,
    f"{RESULTS_DIR}/GBAPD_HC_EUR/GBA_HC_EUR_r11_risk_results_raw.txt",
    label="GBA1-PD vs HC(EUR)"
)

plot_roc(
    ax,
    f"{RESULTS_DIR}/GBAPD_IPD/GBAPD_IPD_r11_risk_results_raw.txt",
    label="GBA1-PD vs IPD(EUR)"
)

ax.set_title("ROC Curves: GBA1-PD vs HC,NMC,and iPD(EUR)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
fig.tight_layout()

fig.savefig(f"{RESULTS_DIR}/GBA1_r11_ROC_raw.png", dpi=300)
plt.close(fig)

In [ ]:
# visualization
# make ROC curves together in one plot

def plot_roc(ax, filepath, label, y="CASE", score="probDisease", lw=2):
    df = pd.read_csv(filepath, sep="\t")
    fpr, tpr, _ = roc_curve(df[y], df[score])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=lw, label=f"{label} (AUC = {roc_auc:.2f})")
    return roc_auc

# --- one plot ---
fig, ax = plt.subplots(figsize=(8, 5))
sns.set_palette("Dark2")  # optional styling

# diagonal
ax.plot([0, 1], [0, 1], color="gray", linestyle="--")

# add each ROC curve
plot_roc(
    ax,
    f"{RESULTS_DIR}/GBAPD_HCcarriers_EUR/GBA_HCcarriers_r11_risk_results_std.txt",
    label="GBA1-PD vs NMC(EUR)"
)

plot_roc(
    ax,
    f"{RESULTS_DIR}/GBAPD_HC_EUR/GBA_HC_EUR_r11_risk_results_std.txt",
    label="GBA1-PD vs HC(EUR)"
)

plot_roc(
    ax,
    f"{RESULTS_DIR}/GBAPD_IPD/GBAPD_IPD_r11_risk_results_std.txt",
    label="GBA1-PD vs IPD(EUR)"
)

ax.set_title("ROC Curves: GBA1-PD vs HC,NMC,and iPD(EUR)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
fig.tight_layout()

fig.savefig(f"{RESULTS_DIR}/GBA1_r11_ROC_std.png", dpi=300)
plt.close(fig)